# GLUE MRPC Near-Duplicate Detection Demo

This notebook demonstrates the **GLUE MRPC dataset** for paraphrase detection (near-duplicate identification).

## What is this?

The **Microsoft Research Paraphrase Corpus (MRPC)** is a benchmark dataset for identifying near-duplicate sentence pairs. It contains 4,076 sentence pairs from news articles, labeled:
- **1** = paraphrase (near-duplicate, high lexical overlap)
- **0** = non-paraphrase (different meanings)

This is useful for MinHash-based near-duplicate detection: paraphrase pairs share high n-gram overlap, making them ideal for evaluating shingling and Jaccard similarity methods.

## Dataset Stats
- **Size**: 4,076 examples
- **Positive rate**: 67.5% (2,753 paraphrases)
- **Source**: GLUE benchmark (Dolan & Brockett 2005; Wang et al. 2019)
- **Domain**: News articles
- **Task**: Binary classification (paraphrase or not)

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Install loguru (not pre-installed on Colab)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import sys
from pathlib import Path
from loguru import logger
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configure logging for notebook (simpler than file-based logging)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper with GitHub URL + local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-cb9424-landmark-pair-fingerprinting-for-text-cr/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        logger.info(f"Loading data from GitHub...")
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.warning(f"GitHub load failed ({e}), trying local file...")
    
    # Fallback to local file
    local_path = Path("mini_demo_data.json")
    if local_path.exists():
        logger.info(f"Loading data from local file...")
        with open(local_path) as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local file")

In [ ]:
# Load the data
data = load_data()
logger.info(f"Loaded data with metadata: {data['metadata']['dataset']}")

## Configuration

Set minimal parameters for a quick demo. Tune these to scale up the processing.

In [ ]:
# Configuration: tunable parameters for the demo
# Start with minimal values, scale up if needed

# Number of examples to process from the dataset
NUM_EXAMPLES = 3  # Start with 3 (all mini data), can increase to 50, 100, etc.

# Shingle configuration for MinHash simulation
SHINGLE_SIZE = 2  # Character n-gram size (2 for bigrams)
NUM_HASHES = 128  # Number of hash functions for MinHash sketch

# Display configuration
DISPLAY_TRUNCATE_LEN = 100  # Truncate long text in output for readability

## Processing

Extract examples from the dataset and prepare for analysis. Standardize to the exp_sel_data_out schema.

In [ ]:
# Extract examples from the dataset
all_datasets = data["datasets"]
examples = []

for dataset_group in all_datasets:
    examples.extend(dataset_group["examples"])

# Limit to NUM_EXAMPLES for the demo
examples = examples[:NUM_EXAMPLES]
logger.info(f"Processing {len(examples)} examples")

In [ ]:
# Parse examples and extract sentence pairs
processed_examples = []

for i, example in enumerate(examples):
    # Parse the input JSON which contains the sentence pair
    input_data = json.loads(example["input"])
    sentence1 = input_data["sentence1"]
    sentence2 = input_data["sentence2"]
    label = int(example["output"])
    
    processed_examples.append({
        "index": example["metadata_row_index"],
        "sentence1": sentence1,
        "sentence2": sentence2,
        "label": label,
        "is_paraphrase": label == 1,
        "source": example["metadata_source"],
    })

logger.info(f"Parsed {len(processed_examples)} examples")

In [ ]:
# MinHash-inspired analysis: compute shingles for each sentence pair
def get_shingles(text, k=SHINGLE_SIZE):
    """Extract k-gram shingles from text (case-insensitive, lowercase)."""
    text = text.lower()
    shingles = set()
    for i in range(len(text) - k + 1):
        shingles.add(text[i:i+k])
    return shingles

def jaccard_similarity(set1, set2):
    """Compute Jaccard similarity between two sets."""
    if len(set1) == 0 and len(set2) == 0:
        return 1.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0.0

# Compute shingles and Jaccard similarity for each pair
analysis_results = []

for ex in processed_examples:
    shingles1 = get_shingles(ex["sentence1"])
    shingles2 = get_shingles(ex["sentence2"])
    jaccard = jaccard_similarity(shingles1, shingles2)
    
    analysis_results.append({
        "index": ex["index"],
        "label": ex["label"],
        "is_paraphrase": ex["is_paraphrase"],
        "s1_length": len(ex["sentence1"]),
        "s2_length": len(ex["sentence2"]),
        "shingle_overlap": len(shingles1 & shingles2),
        "jaccard_similarity": jaccard,
        "sentence1": ex["sentence1"],
        "sentence2": ex["sentence2"],
    })

logger.info(f"Computed shingle analysis for {len(analysis_results)} pairs")

## Results

Summary of key findings and visualization of Jaccard similarity scores by paraphrase label.

In [ ]:
# Create a pandas DataFrame for easy viewing
results_df = pd.DataFrame(analysis_results)

# Summary statistics
print("=" * 80)
print("GLUE MRPC Dataset: Paraphrase Detection via Shingle Analysis")
print("=" * 80)
print(f"\nDataset metadata:")
print(f"  Source: {data['metadata']['source']}")
print(f"  Description: {data['metadata']['description']}")
print(f"  Full dataset size: {data['metadata']['num_rows']} examples")
print(f"  Positive rate (full): {data['metadata']['positive_rate']:.1%}")
print(f"\nDemo sample: {len(results_df)} examples")
print(f"  Paraphrases: {results_df['is_paraphrase'].sum()}")
print(f"  Non-paraphrases: {(~results_df['is_paraphrase']).sum()}")

print(f"\nShingle Analysis (k={SHINGLE_SIZE}):")
print(f"  Mean Jaccard (Paraphrase): {results_df[results_df['is_paraphrase']]['jaccard_similarity'].mean():.3f}")
print(f"  Mean Jaccard (Non-paraphrase): {results_df[~results_df['is_paraphrase']]['jaccard_similarity'].mean():.3f}")
print(f"  Overall mean Jaccard: {results_df['jaccard_similarity'].mean():.3f}")

print("\n" + "=" * 80)
print("Detailed Results:")
print("=" * 80)

# Display detailed results
for idx, row in results_df.iterrows():
    label_str = "PARAPHRASE" if row['is_paraphrase'] else "NON-PARAPHRASE"
    s1_display = row['sentence1'][:DISPLAY_TRUNCATE_LEN] + ("..." if len(row['sentence1']) > DISPLAY_TRUNCATE_LEN else "")
    s2_display = row['sentence2'][:DISPLAY_TRUNCATE_LEN] + ("..." if len(row['sentence2']) > DISPLAY_TRUNCATE_LEN else "")
    
    print(f"\nExample {row['index']} [{label_str}]")
    print(f"  S1: {s1_display}")
    print(f"  S2: {s2_display}")
    print(f"  Jaccard similarity: {row['jaccard_similarity']:.3f}")
    print(f"  Shingle overlap: {row['shingle_overlap']}")
    print(f"  S1 length: {row['s1_length']}, S2 length: {row['s2_length']}")

In [ ]:
# Visualization: Jaccard similarity by paraphrase label
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Box plot of Jaccard similarity by label
label_names = ['Non-Paraphrase\n(Label=0)', 'Paraphrase\n(Label=1)']
jaccard_by_label = [
    results_df[~results_df['is_paraphrase']]['jaccard_similarity'].values,
    results_df[results_df['is_paraphrase']]['jaccard_similarity'].values
]

axes[0].boxplot(jaccard_by_label, labels=label_names)
axes[0].set_ylabel('Jaccard Similarity', fontsize=11)
axes[0].set_title(f'Jaccard Similarity by Label (k={SHINGLE_SIZE})', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# Plot 2: Bar chart of mean Jaccard by label
mean_jaccard = [
    results_df[~results_df['is_paraphrase']]['jaccard_similarity'].mean(),
    results_df[results_df['is_paraphrase']]['jaccard_similarity'].mean()
]
colors = ['#ff7f0e', '#1f77b4']
bars = axes[1].bar(label_names, mean_jaccard, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Mean Jaccard Similarity', fontsize=11)
axes[1].set_title(f'Mean Jaccard Similarity by Label (k={SHINGLE_SIZE})', fontsize=12, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nVisualization complete!")